# MATH 189 Final Report

The following is a general checklist for what to include in the final report.

* A statement of the problem you are investigating
* Why is this problem relevant? Or, what inspired you to investigate this problem?
* Where did you get the data from?
* Description of the data
* Exploratory data analysis
* What (if any) analyses have already been performed on this data (or another similar dataset)?
    * You should provide references to this
* What types of analyses did you perform?
* How do you interpret the results from these analyses?
* What are some potential limitations and shortcoming of your analyses?

**Members: Charlene Hsu, Beau Luc, Havyn Nguyen, Naomi Metzler**

## Abstract

...

<hr>

## Problem Statement

### Proposal Question

Are census districts with higher SNAP participation more likely to experience low supermarket access, and does this relationship differ between urban and rural areas?

### Problem Relevance & Background

Food insecurity has always been an issue that has been eating away at many hungry Americans each year, proving to have be a systematic American issue. One such measure to mitigate the issue by the government was implementing various programs such as our focal point, SNAP. Alongside Electronic Benefit Transfer (EBT) being standardized the previously known Food Stamp Act of 1977 was appropriately renamed as the Supplemental Nutrition Assistance Program better known as SNAP of Oct. 1, 2008. SNAP provides its recipients with an EBT card that can be used at select stores to purchase food ranging from fruits, vegetables, frozen foods, baby formula, and more. SNAP as a program served as an useful indicator and estimator of income and poverty throughout the countries and school districts. UCSD Basic Needs office has reported that 16,716 UCSD students during 2022 to 2023 were direct beneficiaries of CalFresh, California’s SNAP program, demonstrating the need to observe the effectiveness and reachability of the program. Furthermore, this illustrates that the problem of food insecurity is one not stopping in just our university but an issue that is constantly being faced by millions of Americans across the countries. Which is why proper data collection and analysis pertaining to the SNAP participation can shine a light on how funds should be distributed and how well the program is managed.

In connection to why analyzing SNAP participation and the geographic infrastrucutres of supermarket access between urban and rural areas, we wanted to be able to find associations and be able to determine why there are still food desert concerns even with government issued programs that are in place to fight these issues, in an essence of understanding why it is a problem if SNAP participation is high but access is low. This indicates that food deserts not only are in relation to the income of an area or poverty levels, but rather the concept of a food mirage—areas where grocery stores exist, but the food is economically inaccessible to low-income residents or individuals who lack the physical infrastructure to redeem SNAP benefits efficiently. Rural low-access areas are typically defined by larger physical distances between physical infrastructures and a lack of public transportation, but in contrast, urban areas may have more densely packed infrastructure, but many lack vehicle ownership or even simply the time. When high-SNAP areas lack supermarkets, recipients spend more of their costs on transportation, effectively reducing the purchasing power of their SNAP benefits.

<hr>

## Datasource

#### Food Access Research Atlas

With this topic, the data source that will be used is from the Food Access Research Atlas, which includes data that is aggregated into an Excel spreadsheet. This dataset provides a summary of food access metrics for both low-income and other census tracts using multiple measures of supermarket accessibility and includes information on food access for populations living within each census tract. The Food Access Research Atlas used data of food access estimates from 2015 to 2019, where the estimates are derived from a 2019 supermarket database, the 2010 Decennial Census, and the 2014–2018 American Community Survey (ACS), while the 2015 estimates rely on a 2015 supermarket database, the 2010 Census, and the 2010–2014 ACS. The Food Access Research Atlas provides multiple measures of food accessibility across census tracts by evaluating residents’ proximity to supermarkets and access to transportation. In urban areas, food access is measured using ½-mile and 1-mile distance thresholds to the nearest supermarket, while rural areas use 10-mile and 20-mile thresholds due to lower population density and greater travel distances. The Atlas also incorporates vehicle availability for all census tracts, allowing users to examine how transportation access influences food accessibility.

### Data Cleaning

> **Row / Column Counts**

| Stage | Rows | Columns |
|---|---|---|
| Raw | 72,531 | 147 |
| After cleaning (all tracts) | 72,531 | 159 |
| Modeling subset | 42,156 | 26 |

No rows were dropped in the full cleaned dataset. Percentage variables with impossible values (below 0 or above 100, caused by denominator issues) were set to NaN and documented below.

The modeling subset dropped **30,375 rows** missing at least one required variable:
`pct_low_income_low_access`, `pct_snap`, `PovertyRate`, `MedianFamilyIncome`, `pct_no_vehicle`, or `Urban`. These tracts either have zero population/housing units or are missing key USDA measurements.

The data covers SNAP participation, income, vehicle access, and supermarket proximity, which are specific data that we need for our problem statement.

> **Response and Explanatory Variables**

**Main response variable:** `pct_low_income_low_access`  
Definition: The percentage of residents in a census tract who are both low-income and low-access to supermarkets (beyond 1 mile in urban areas, 10 miles in rural areas).  
* Formula: `100 × LALOWI1_10 / Pop2010`

**Main explanatory variable:** `pct_snap`  
Definition: The percentage of occupied housing units in a census tract that receive SNAP benefits.  
* Formula: `100 × TractSNAP / OHU2010`

> **Variables Created**

In this step, the following variables were feature engineered to transform the raw data into more usable and informative data for our problem statement. Variables were created from existing ones from the data to capture underlying patterns. These featured engineered variables are made up of computed percentage variables from raw counts (e.g. % of households on SNAP), variables that have been safely handled for zero denominators so no invalid values carried through, flagged 74 out-of-range values as missing rather than deleting rows, and log-transformed income and the response variable to reduce skew.

| New Variable | Formula | Notes |
|---|---|---|
| `pct_snap` | `100 × TractSNAP / OHU2010` | Main explanatory variable |
| `pct_no_vehicle` | `100 × TractHUNV / OHU2010` | No-vehicle households |
| `pct_low_income_low_access` | `100 × LALOWI1_10 / Pop2010` | Main response variable |
| `pct_children` | `100 × TractKids / Pop2010` | |
| `pct_seniors` | `100 × TractSeniors / Pop2010` | |
| `pct_white` | `100 × TractWhite / Pop2010` | |
| `pct_black` | `100 × TractBlack / Pop2010` | |
| `pct_asian` | `100 × TractAsian / Pop2010` | |
| `pct_hispanic` | `100 × TractHispanic / Pop2010` | |
| `urban_label` | `"Urban"` if `Urban==1`, else `"Rural"` | |
| `log_median_family_income` | `log(MedianFamilyIncome)` | NaN when ≤ 0 |
| `log_response` | `log1p(pct_low_income_low_access)` | Stabilizes right skew |

All percentage variables use safe division: denominator values ≤ 0 or NaN return NaN instead of crashing or producing Inf.

> **Missing Value Handling**

From this point, our data started with 72,531 tracts;, where we ended keeping 42,156 with complete required variables list previously. With this, the next step was handling our missing values, which we did by dropping tracts that were mostly unpopulated or missing key measurements.

- **Pop2010:** 0 missing; used as denominator for population-based percentages.
- **OHU2010:** 106 rows have zero/missing; `pct_snap` and `pct_no_vehicle` are NaN for those rows.
- **Impossible percentage values** (outside [0, 100]) were set to NaN:
  - `pct_snap`: 21 rows
  - `pct_no_vehicle`: 52 rows
  - `pct_low_income_low_access`: 1 row
- Rows with any missing value in the required modeling columns were excluded from the modeling dataset only (30,375 rows).

> **Outlier Handling**

Outliers were flagged and documented but not removed. Below are the variables that contained outliers.

| Variable | Fence (low, high) | Outlier count |
|---|---|---|
| `pct_snap` | [−16.55, 39.91] | 3,023 |
| `pct_no_vehicle` | [−10.19, 23.63] | 6,409 |
| `pct_low_income_low_access` | [−18.92, 34.92] | 2,741 |

Outliers are **not removed**. Because all three variables are non-negative, the lower IQR fence is meaningless (values cannot go below 0). Only the upper-tail counts are substantively relevant. Inspect high-value tracts before modeling; consider log transformation or robust regression if residuals are skewed.

#### Important Data Cleaning Choices & Potential Limitations

With the above walk-through of our data cleaning process and handling, here are some important data cleaning choices. In summary, we have kept all 72,531 tracts in the full cleaned file, additionally, any zero or missing denominators produced missing values instead of outputing as errors or dropped rows. Out-of-range percentages were flagged as missing and documented and not silently deleted. State and county columns preserved to help account for geographic clustering later in our regression modeling. And, outliers were documented but not removed, where it was determined that these extreme values may reflect real need and not bad data.

Some limitations that could come from our data cleaning process for our regression model would be that 42% of tracts are excluded, which are mostly zero-population tracts, such as parks, airports, or industrial areas. Neighboring tracts are spatially correlated, which are only partially addressed by state fixed effects. Another big limitation is that the data is from 2019 and may not reflect post-pandemic shifts in SNAP participation.


<hr>

## Exploratory Data Analysis

With this we used the set up Modeling dataset that contains 42,156 rows x 26 columns for our Exploratory Data Analysis. In our EDA, the following are key variables that we are working with to help us draw conclusions for our problem statement regarding SNAP participation and grocery access in urban versus rural areas:

| Variable | Description |
|---|---|
| `pct_low_income_low_access` | % of tract residents who are both low-income and have low supermarket access *(response)* |
| `pct_snap` | % of occupied housing units (HUs) receiving SNAP *(main predictor)* |
| `pct_no_vehicle` | % of HUs without a vehicle |
| `Urban` | 1 = urban tract, 0 = rural tract |
| `PovertyRate` | Tract-level poverty rate (%) |
| `log_median_family_income` | Median family income as its log transformation |
| `pct_children`, `pct_seniors` | Age controls |
| `pct_black`, `pct_hispanic`, `pct_asian` | Racial controls (reference: % White) |

<br>

> **Distribution of `pct_low_income_low_access` (Response Variable)**

The distribution of `pct_low_income_low_access` is strongly right-skewed, where most tracts have low LILA rates, but there is a long upper tail of high-need tracts that exists. This suggests log transformation may be appropriate for regression. With this, a log1p transformation is performed that reduces skewness from 2.04 to essentially zero and brings the Q-Q plot closer to the diagonal, which confirms `log_response` is used as the dependent variable in all regression models.

![01_hist_response.png](attachment:01_hist_response.png)

Figure 1.

<br>

> **Distribution of `pct_snap`**

`pct_snap` is the main explanatory variable and based on analysis of its distribution we noticed that is right-skewed, which translates to SNAP participation is right-skewed. The majority of tracts have under 20% SNAP uptake, but a meaningful number of tracts — likely high-poverty urban and rural areas — have substantially higher rates.

![02_hist_pct_snap.png](attachment:02_hist_pct_snap.png)

Figure 2.

<br>


> **SNAP vs LILA Relationship**

There is a positive association between SNAP participation and low-income low-access rates. Urban tracts (displayed in blue) tend to cluster at lower low-access values despite high SNAP rates, while rural tracts (displayed in red) show more spread at higher low-access values, suggesting the SNAP–access relationship may differ by urban/rural status, which directly motivates our interaction model.

![03_scatter_snap_vs_lila.png](attachment:03_scatter_snap_vs_lila.png)

Figure 3.

<br>

> **Urban vs Rural Baseline**

Displayed is a boxplot of the response by urban or rural tracts, where it can be seen that urban tracts have higher median LILA rates than rural tracts even in raw data before controlling for other variables like income, racial distribution, etc. This confirms that including `Urban` as a covariate (and interaction term) in regression is important. The urban mean is 11.0%, while the rural mean is 8.4%, which in this case, may indicate that the urban mean may take into outliers within the urban tract, therefore looking at the boxplot confirms that `Urban` must be included as a covariate (not just an interaction) to control for the baseline difference in food access between the two tract types.

![04_boxplot_lila_by_urban.png](attachment:04_boxplot_lila_by_urban.png)

Figure 4.

<br>

> **No-vehicle households vs. Low-income Low-access**

Vehicle access shows a moderate positive association with low-income low-access rates in rural tracts. In urban tracts the pattern is weaker, which is consistent with urban areas having more public transit alternatives. This motivates the `pct_no_vehicle:Urban` interaction term.

![05_scatter_noveh_vs_lila.png](attachment:05_scatter_noveh_vs_lila.png)

Figure 5.

<br>

> **Correlation Heat Map**

Based on the pairwise correlation heat map, `PovertyRate` and `log_median_family_income` correlate at -0.80, which is strong enough to cause multicoliniearity if both are included in a single model, so models use one or the other, which we will apply later on. It can also be seen that `pct_snap` has a moderate positive correlation with the response variable, but income and poverty are more strongly correlated as seen, therefore, robust checks will be performed, one using log income (Model A) and one using poverty rate (Model B).

![06_correlation_heatmap.png](attachment:06_correlation_heatmap.png)

Figure 6.

<br>

> **Urban vs Rural Distributions**

Welche's two-sample t-tests were performed to compare urban vs rural distrubutions for each of the five key variabes, where all five key variabes, which are the SNAP rate (`pct_snap`), the no-vehicle rate (`pct_no_vehicle`), LILA rate (`pct_low_income_low_access`), poverty rate, and log income (`log_median_family_income`) are significantly different between urban and rural tracts, where all two-sample t-tests p-values < 0.001. Urban tracts have higher SNAP participation and higher LILA rates on average, motivating inclusion of `Urban` as a predictor and interaction term.

![07_urban_rural_distributions.png](attachment:07_urban_rural_distributions.png)

Figure 7.

<br>

> **log Transformation Justification**

The raw response is heavily right-skewed, with a skewness of 2.04. To combat this, the log1p brings it much closer to symmetric, with a log1p-transformed skewness of −0.002. The Q-Q plot of raw response shows extreme nonnormality, log-transformed version is closer to the linear diagonal, with this, a conclusion that all regression models will use `log_response` as the dependent variable.

![08_log_transform.png](attachment:08_log_transform.png)

Figure 8.

<br>

> **SNAP vs LILA Relationship Across Income Quartiles**

Pearson correlation tests were performed for `pct_snap` vs `pct_low_income_low_access` for each of the 4 income quartiles. The positive SNAP-LILA correlation holds in all four income quartiles (all $r>0, p<0.001$). This association is not simply a result of income, rather that SNAP correlates with LILA across the income distribution.

![09_snap_lila_by_income_quartile.png](attachment:09_snap_lila_by_income_quartile.png)

Figure 9.

<br>

> **Racial Composition and Food Access**

The following graph displays low-access rates by racial majority of tract, where it can be seen that tracts that are majority Black or Hispanic have higher mean LILA rates than tracts that are majority White or Asian. So, we included racial composition as a control in all of our regression models.

![10_demographics_lila.png](attachment:10_demographics_lila.png)

Figure 10.



<hr>

## Statistical Analysis

### Proposed Analysis

To examine whether census tracts with higher SNAP participation are more likely to experience low supermarket access, and whether this relationship differs between urban and rural areas, we will use multivariable regression. We will construct proportions rather than use raw counts, so that census tracts with different population sizes can be compared more appropriately. The response variable will be the rate of low access, measured as the proportion of the tract population living beyond 1 mile in urban areas or 10 miles in rural areas from a supermarket (LAPOP1_10 divided by POP2010). The main independent variable will be tract SNAP participation rate, calculated as TractSNAP divided by OHU2010 which we defined as pct_snap. We will include poverty rate, median family income, and tract proportion of housing units without vehicles as control variables, since these all may influence both SNAP participation and supermarket access. We will include urban vs rural classification using the Urban flag variable.
Before creating the regression model, we will carry out exploratory data analysis and check for multicollinearity among the predictor variables using correlation matrices and Variance Inflation Factors (VIF). Then, we’ll fit a multiple linear regression model with an interaction term between SNAP participation rate and the Urban indicator variable to test whether the association between SNAP participation and low supermarket access differs between urban and rural census areas. We’ll verify model assumptions using residual plots, Q-Q plots, and Shapiro-Wilk test, and check for outliers and influential observations.
Lastly, we will perform hypothesis tests on the fitted model. We will use an F-test to determine whether the model is statistically significant as a whole. A t-test on the SNAP participation coefficient will determine whether SNAP participation is associated with low supermarket access after controlling for poverty, income, transportation access, and urban/rural classification. We’ll use a second t-test on the interaction term to figure out whether the relationship between SNAP participation and low supermarket access differs significantly between urban and rural areas, addressing the second part of our research question.

### Analysis

As discussed in the EDA section, we compared urban vs rural distributions for each of the 5 key variables with Welch's two-sample t-tests and used histograms, scatter plots, box plots, QQ-plots, and correlation heat maps, to visualize relationships between variables in EDA. With this we can discuss the following model set up based on our understanding of our data and relationships between variables.

> **Model Set Up**
> * Three OLS (ordinary least squares) regression models are fit with dependent variable `log_response`
> * standard errors clustered at the state level to account for spatial autocorrelation within state
> * pct_white is excluded as reference category for racial composition, LILATracts_* variables excluded to avoid data leakage since they are derived from the response, raw counts like TractSNAP and TractHUNV are replaced by percentage versions


#### Model A (main effects with income)
```python
formula_A = (
    'log_response ~ pct_snap + log_median_family_income'
    ' + pct_no_vehicle + Urban'
    ' + pct_children + pct_seniors'
    ' + pct_black + pct_hispanic + pct_asian'
)
```


#### model B (main effects with poverty rate)
* same as model A except replaces log_median_family_income with PovertyRate

```python
formula_B = (
    'log_response ~ pct_snap + PovertyRate'
    ' + pct_no_vehicle + Urban'
    ' + pct_children + pct_seniors'
    ' + pct_black + pct_hispanic + pct_asian'
)
```

* pct_snap becomes significant ($\beta = 0.010, p<0.001$) meaning poverty rate doesn't absorb the SNAP signal as much as income
* shows socioeconomic disadvantage predicts food access gaps with either measure (income vs poverty rate)


#### model C (interaction model / primary model)
* extends model A, adds interaction terms pct_snap:Urban and pct_no_vehicle:Urban

```python
formula_C = (
    'log_response ~ pct_snap + log_median_family_income'
    ' + pct_no_vehicle + Urban'
    ' + pct_snap:Urban + pct_no_vehicle:Urban'
    ' + pct_children + pct_seniors'
    ' + pct_black + pct_hispanic + pct_asian'
)
```

* tests whether SNAP-LILA association is different for urban vs rural tracts
* lowest AIC of the three models
* F-test shows interaction terms are together significant 


* For each model, z-tests on individual coefficients (reported in each model summary as z-statistics and p-values)
* F-test (model A vs model C) to test whether the two interaction terms jointly imporve model fit

# add stuff about assumptions and assumptions checks that Beau did

<hr>

## Results

#### urban vs rural gap
* Urban tracts have substantially higher low access rates, even after controlling for income, poverty, and demographics 
* Urban coefficient = +0.635 (p < 0.001), corresponding to ~89% higher LILA rates relative to rural tracts


#### SNAP participation (model C )
* In rural tracts (Urban = 0): higher SNAP is associated with lower log-LILA rates ($\beta = −0.015$, p = 0.011)
* In urban tracts: net SNAP effect = −0.015 + 0.020 = +0.005 higher SNAP is associated with higher log-LILA rates
* Interaction term pct_snap:Urban = +0.020 (p = 0.001): urban vs rural difference in the SNAP effect is statistically significant


#### Income
* log_median_family_income is the strongest predictor across all models ($\beta = −0.859$ in model C, $p < 0.001$)
* Higher median family income is strongly associated with lower low access rates across both urban and rural tracts


#### Poverty rate (model B)
* PovertyRate = +0.027 (p < 0.001): higher poverty rate is associated with higher LILA rates
* consistent with income in model A - socioeconomic disadvantage is associated with lower food access using either measurement


#### Vehicle access
* pct_no_vehicle:Urban = −0.044 (p < 0.001): the negative effect of lack of vehicle access is significantly weaker in urban tracts, which makes sense with urban areas having greater access to public transit and walkable stores


#### Model fit
* Model C explains ~13.5% of variance in log-LILA (adjusted R-squared = 0.135)
* Model A adjusted R-squared = 0.132, Model B adjusted R-squared = 0.117



<hr>

## Limitations & Shortcomings

Our models A,B,and C have a $R^2$ of $0.1325,0.1170 \text{, and } 0.1355$ respectatively. This implies that our model explain only a fraction of the variance from $11.7\%$ to $13.55\%$ which could indicate that there were other factors that could be influential that the data didn't capture. Despite this the low $R^2$ value shouldn't raise too much concern as examining the SNAP census tract population is bound to be harder to predict with the many factors that can attributed to how people use their SNAP benefit and where. Additionally, our diagnosis reveals that our data violates homoscedasticity and normality conditions. We are still able to trust our result from our data analysis due to the large sample size of 42,156 census tract which means the normality conditions is not as a considerable issue. In the issue of homosecdasticity, we address this issue in model C since we applied cluster standard error to reduce the impact of this condition violation. Thus, our best model in interpreting coefficients and factors is model C which additionally has the highest $R^2$ of $13.55\%$ so we able to extract information with careful consideration. 

### Future Steps

In the future, we would like our data to include more factors such as grocery delivery and alternative means of transportation such our data is from 2019. Additionally, we should note that 2019 is the year just before the COVID-19 pandemic so future data would heavily be impacted by such a global event. Especially, since our data examines the variable of no vehicle access to be such a heavy factor then with the pandemic having people stay indoor and traveling less. Then, we should expect this factor to surge in importance. Thus, if we were to examine a more recent dataset from 2020 to 2026 then we should compare with a dataset from a time period where the pandemic is less prominent if we want to apply our results. 